In [ ]:
# ==============================================================================
# PARTE 2: MODELO DE REGRESIÓN Y EVALUACIÓN CON C-MAPSS (FD001)
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CARGA DE DATOS DE ENTRENAMIENTO (YA LIMPIO)
# ------------------------------------------------------------------------------
df_train = pd.read_csv('FD001Train_Clean.csv')

# Separar variables predictoras y variable objetivo
X_train = df_train.drop(columns=['RUL'])
y_train = df_train['RUL']

# 2. CARGA Y LIMPIEZA DEL SET DE TEST
# ------------------------------------------------------------------------------
# Leer archivo test
df_test_raw = pd.read_csv('test_FD001.txt', sep='\s+', header=None)
#df_test_raw.drop(columns=[26,27], inplace=True)

column_names = [
    'unit_number', 'time_in_cycles', 'op_set1', 'op_set2', 'op_set3',
    'sensor1', 'sensor2', 'sensor3', 'sensor4', 'sensor5',
    'sensor6', 'sensor7', 'sensor8', 'sensor9', 'sensor10',
    'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15',
    'sensor16', 'sensor17', 'sensor18', 'sensor19', 'sensor20', 'sensor21'
]
df_test_raw.columns = column_names

# 2b. ALINEACIÓN DE SENSORES ENTRE TRAIN Y TEST
# ------------------------------------------------------------------------------

# Obtenemos las columnas predictoras usadas en el entrenamiento (excluimos identificadores)
cols_predictoras = [col for col in X_train.columns if col not in ['unit_number', 'time_in_cycles']]

# Nos aseguramos de que el conjunto de test tenga SOLO esas columnas y en el MISMO orden
df_test_predictoras = df_test_raw[cols_predictoras].copy()

# 2c. PARA EL SET DE TEST, SOLO USAMOS LA ÚLTIMA FILA DE CADA MOTOR
# Cada unidad (motor) en test tiene múltiples ciclos, pero el RUL real solo está disponible para el último ciclo.
X_test = df_test_raw.groupby('unit_number').last().reset_index()
X_test_predictoras = X_test[cols_predictoras]

# 3. CARGA DE ARCHIVO DE RUL REAL PARA TEST
# ------------------------------------------------------------------------------
y_test_true = pd.read_csv('RUL_FD001.txt', header=None).squeeze()

# 4. ENTRENAMIENTO DE MODELOS Y PREDICCIÓN
# ------------------------------------------------------------------------------

# Regresión Lineal
model_lr = LinearRegression()
model_lr.fit(X_train[cols_predictoras], y_train)
y_pred_lr = model_lr.predict(X_test_predictoras)

# Random Forest
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train[cols_predictoras], y_train)
y_pred_rf = model_rf.predict(X_test_predictoras)

# 5. EVALUACIÓN DE MODELOS
# ------------------------------------------------------------------------------
def evaluar(y_true, y_pred, nombre_modelo="Modelo"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"\nResultados para {nombre_modelo}:")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE : {mae:.2f}")
    print(f"  R2  : {r2:.2f}")
    return rmse, mae, r2

_ = evaluar(y_test_true, y_pred_lr, nombre_modelo="Regresión Lineal")
_ = evaluar(y_test_true, y_pred_rf, nombre_modelo="Random Forest")

# 6. VISUALIZACIÓN DE RESULTADOS
# ------------------------------------------------------------------------------
plt.figure(figsize=(14,6))
plt.plot(y_test_true.values, 'o-', label="RUL Real (Test)")
plt.plot(y_pred_lr, 's-', label="Predicción Regresión Lineal")
plt.plot(y_pred_rf, 'x-', label="Predicción Random Forest")
plt.xlabel("ID de unidad (orden test)")
plt.ylabel("RUL")
plt.title("Comparación de RUL real vs predicho")
plt.legend()
plt.tight_layout()
plt.show()

# Scatter plot para ver ajuste
plt.figure(figsize=(8,8))
plt.scatter(y_test_true, y_pred_rf, alpha=0.7, label="Random Forest")
plt.scatter(y_test_true, y_pred_lr, alpha=0.7, label="Linear Regression", marker="s")
plt.plot([y_test_true.min(), y_test_true.max()], [y_test_true.min(), y_test_true.max()], 'k--', label="Ideal")
plt.xlabel("RUL Real")
plt.ylabel("RUL Predicho")
plt.title("Predicción de RUL (Test) - Comparación")
plt.legend()
plt.tight_layout()
plt.show()

# ==============================================================================
# FIN DEL SCRIPT DE REGRESIÓN Y EVALUACIÓN EN C-MAPSS FD001
# ==============================================================================
